In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
PATH_DATA_RAW = "../data/raw/"
PATH_DATA_CLEAN = "../data/clean/"

# Model time index: t = 1 corresponds to 1949
STARTYEAR = 1949

In [18]:
# 1. Load raw CSVs

# NOTE: -999, -66, -77 are CIRI's sentinel codes for missing/unclassifiable
ciri_df = read.csv(paste0(PATH_DATA_RAW, "CIRI_physint_data20120401.csv"), na.strings = c(-999, -66, -77))
pts_df = read.csv(paste0(PATH_DATA_RAW, "PTS2011.csv"))
hathaway_df = read.csv(paste0(PATH_DATA_RAW, "Hathaway2002longData.csv"))

# NOTE: ITT uses string sentinels, so na.strings must be quoted
itt_df = read.csv(paste0(PATH_DATA_RAW, "ITT_CY.csv"), na.strings = c("-999", "-888", "-777"))

genocide_df = read.csv(paste0(PATH_DATA_RAW, "Genocide_Politicide_panel2010.csv"))
rummel_df = read.csv(paste0(PATH_DATA_RAW, "Rummel_Politicide_panel1987.csv"))
killing_df = read.csv(paste0(PATH_DATA_RAW, "OneSidedKilling1989_2011.csv"))
whpsi_df = read.csv(paste0(PATH_DATA_RAW, "whpsi.csv"))
massive_df = read.csv(paste0(PATH_DATA_RAW, "Massive_State_Repression_Panel_1988.csv"))

cat("Loaded 9 datasets. Num of rows:\n")
cat(sprintf("   CIRI: %d \n   PTS: %d \n   Hathaway: %d \n   ITT: %d\n",
            nrow(ciri_df), nrow(pts_df), nrow(hathaway_df), nrow(itt_df)))
cat(sprintf("   Genocide: %d \n   Rummel: %d \n   Killing: %d \n   WHPSI: %d \n   Massive: %d\n",
            nrow(genocide_df), nrow(rummel_df), nrow(killing_df), nrow(whpsi_df), nrow(massive_df)))

Loaded 9 datasets. Num of rows:
   CIRI: 6030 
   PTS: 6228 
   Hathaway: 2460 
   ITT: 1672
   Genocide: 8594 
   Rummel: 4868 
   Killing: 244 
   WHPSI: 4035 
   Massive: 5409


In [19]:
# 2. Harmonize CIRI COW codes
# CIRI has redundant observations where COW treats split/successor states as a
# single unit (e.g. Yugoslavia -> Serbia, USSR -> Russia). Drop duplicates and
# reassign COW codes to match the rest.

# Pre-assign Serbia's COW code before subsetting so the row filter below works
ciri_df$COW[ciri_df$CIRI == 560] = 345

# Keep only the COW-canonical country-years for each successor/predecessor pair:
#   530 = Russia (post-1991), 590 = Soviet Union (pre-1992)
#   560 = Serbia (post-2006), 458 = Montenegro (post-2006)
#   689 = Yugoslavia (pre-1992), 563 = Serbia & Montenegro (1992-2005)
#   692 = Yugoslavia Federal Republic (2000-2002 only)
ciri_df = subset(ciri_df, ciri_df$CIRI != 530 | ciri_df$YEAR >= 1992)
ciri_df = subset(ciri_df, ciri_df$CIRI != 590 | ciri_df$YEAR <= 1991)
ciri_df = subset(ciri_df, ciri_df$CIRI != 560 | ciri_df$YEAR >= 2006)
ciri_df = subset(ciri_df, ciri_df$CIRI != 458 | ciri_df$YEAR >= 2006)
ciri_df = subset(ciri_df, ciri_df$CIRI != 689 | ciri_df$YEAR <= 1991)
ciri_df = subset(ciri_df, ciri_df$CIRI != 563 | ciri_df$YEAR >= 1992 & ciri_df$YEAR <= 1999 |
                                                ciri_df$YEAR >= 2003 & ciri_df$YEAR <= 2005)
ciri_df = subset(ciri_df, ciri_df$CIRI != 692 | ciri_df$YEAR >= 2000 & ciri_df$YEAR <= 2002)

# Reassign COW codes for rows that survived the filters above
n = nrow(ciri_df)
t = 1
while (t <= n) {
  if      (ciri_df$CIRI[t] == 458 && ciri_df$YEAR[t] >= 2006) { ciri_df$COW[t] = 341; t = t + 1 }
  else if (ciri_df$CIRI[t] == 560 && ciri_df$YEAR[t] >= 2006) { ciri_df$COW[t] = 345; t = t + 1 }
  # NOTE: Serbia & Montenegro 2005 row gets Serbia's COW (345) to bridge the gap to 2006+ rows
  else if (ciri_df$CIRI[t] == 563 && ciri_df$YEAR[t] == 2005) { ciri_df$COW[t] = 345; t = t + 1 }
  else t = t + 1
}
rm(t)

# Keep the 4 physical integrity indicators; drop rows with all 4 missing
ciri_df = subset(ciri_df, select = c(YEAR, CIRI, COW, DISAP, KILL, POLPRIS, TORT))
ciri_df = subset(ciri_df, !is.na(DISAP) | !is.na(KILL) | !is.na(POLPRIS) | !is.na(TORT))

cat(sprintf("CIRI after harmonization: %d rows, %d countries\n",
            nrow(ciri_df), length(unique(ciri_df$COW))))

print(head(ciri_df))

CIRI after harmonization: 4751 rows, 197 countries
  YEAR CIRI COW DISAP KILL POLPRIS TORT
1 1981  101 700     0    0       0    0
2 1982  101 700     0    0       0    0
3 1983  101 700     0    0       0    0
4 1984  101 700     0    0       0    0
5 1985  101 700     0    0       0    0
6 1986  101 700     0    0       0    0


In [20]:
# 3. Clean standards-based datasets: PTS, Hathaway, ITT

pts_df = subset(pts_df, select = c(Country, COW., Year, Amnesty, State.Dept.))
names(pts_df) = c("Country", "COW", "YEAR", "Amnesty", "State")

# Drop 2011: PTS file includes it but CIRI only goes to 2010
pts_df = subset(pts_df, YEAR < 2011)

# Same USSR/Russia and Yugoslavia/Serbia splits as CIRI above
pts_df$COW[pts_df$Country == "USSR"] = 999
pts_df = subset(pts_df, pts_df$COW != 365 | pts_df$YEAR >= 1992)
pts_df = subset(pts_df, pts_df$COW != 999 | pts_df$YEAR <= 1991)
pts_df$COW[pts_df$Country == "USSR"] = 365

pts_df$COW[pts_df$Country == "Serbia"] = 999
pts_df = subset(pts_df, pts_df$COW != 345 | pts_df$YEAR <= 2006)
pts_df = subset(pts_df, pts_df$COW != 999 | pts_df$YEAR >= 2007)
pts_df$COW[pts_df$Country == "Serbia"] = 345

pts_df = subset(pts_df, !is.na(Amnesty) | !is.na(State), select = c(-Country))

hathaway_df = subset(hathaway_df, select = c(ccode, year, torture))
names(hathaway_df) = c("COW", "YEAR", "hathaway")

# NOTE: na.omit needed because wide-to-long conversion introduced NAs for
# country-years where the state did not yet exist
hathaway_df = na.omit(hathaway_df)

# Recode ITT text categories to ordered numeric (1 = no allegations ... 6 = systematic)
itt_df$COW = itt_df$cowccode
itt_df$COW[itt_df$COW == 340] = 345  # Serbia
itt_df$COW[itt_df$COW == 678] = 679  # Yemen
itt_df$COW[itt_df$COW == 665] = 666  # Israel/Palestine

itt_df$scale = NA
itt_df$scale[as.character(itt_df$LoT) == "No Allegations"] = 1
itt_df$scale[as.character(itt_df$LoT) == "Infrequent"] = 2
itt_df$scale[as.character(itt_df$LoT) == "Several"] = 3
itt_df$scale[as.character(itt_df$LoT) == "Routinely"] = 4
itt_df$scale[as.character(itt_df$LoT) == "Widespread"] = 5
itt_df$scale[as.character(itt_df$LoT) == "Systematic"] = 6

itt_df$restricted = NA
itt_df$restricted[as.character(itt_df$RstrctAccess) == "No"] = 0
itt_df$restricted[as.character(itt_df$RstrctAccess) == "Yes"] = 1

itt_df = subset(itt_df, select = c(year, COW, scale, restricted))
names(itt_df) = c("YEAR", "COW", "ITT", "ITTrestricted")

cat(sprintf("PTS: %d rows | Hathaway: %d rows | ITT: %d rows\n",
            nrow(pts_df), nrow(hathaway_df), nrow(itt_df)))

print(head(itt_df))

PTS: 5805 rows | Hathaway: 2233 rows | ITT: 1672 rows
  YEAR COW ITT ITTrestricted
1 1995   2   5             1
2 1996   2   5             1
3 1997   2   5             1
4 1998   2   6             1
5 1999   2   6             0
6 2000   2   6             0


In [21]:
# 4. Clean event-based datasets
# Each becomes a binary indicator: 1 if event occurred, 0 otherwise.
# Structural zeros (no event within coverage window) are filled after merging.

genocide_df = subset(genocide_df, select = c(ccode, year, genocide))
names(genocide_df) = c("COW", "YEAR", "genocide")

rummel_df = subset(rummel_df, year >= STARTYEAR)
names(rummel_df) = c("COW", "YEAR", "rummel")

# NOTE: the raw killing file is sparse — only country-years WITH killings appear;
# killing = 1 is added here and implicit zeros are filled after merging
killing_df = subset(killing_df, select = c(ccode, YEAR))
killing_df = subset(killing_df, YEAR < 2011)
names(killing_df) = c("COW", "YEAR")
killing_df$killing = 1

# WHPSI records event counts; we only need a binary indicator
whpsi_df$executions = 0
whpsi_df$executions[whpsi_df$POLITICAL.EXECUTION > 0] = 1
whpsi_df = subset(whpsi_df, YEAR >= STARTYEAR, select = c(COW, YEAR, executions))

massive_df = subset(massive_df, year >= STARTYEAR)
names(massive_df) = c("COW", "YEAR", "massive_repression")

cat(sprintf("Genocide: %d | Rummel: %d | Killing: %d | WHPSI: %d | Massive: %d\n",
            nrow(genocide_df), nrow(rummel_df), nrow(killing_df), nrow(whpsi_df), nrow(massive_df)))

print(head(massive_df))

Genocide: 8594 | Rummel: 4868 | Killing: 234 | WHPSI: 3964 | Massive: 5029
   COW YEAR massive_repression
7    2 1949                  0
8    2 1950                  0
9    2 1951                  0
10   2 1952                  0
11   2 1953                  0
12   2 1954                  0


In [22]:
# 5. Merge all datasets into one country-year panel
# Genocide anchors the merge; each join is outer so no country-year is lost
# just because it is missing from one source.

panel_df = merge(genocide_df, ciri_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = FALSE)
panel_df = merge(panel_df, pts_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, hathaway_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, itt_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, rummel_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, killing_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, whpsi_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)
panel_df = merge(panel_df, massive_df, by = c("COW", "YEAR"), all.x = TRUE, all.y = TRUE)

# Drop ITTrestricted — not used in the main model
panel_df = subset(panel_df, select = c(YEAR, CIRI, COW,
                                       DISAP, KILL, POLPRIS, TORT,
                                       Amnesty, State, hathaway, ITT,
                                       genocide, rummel, massive_repression,
                                       executions, killing))

cat(sprintf("Merged panel: %d rows, %d columns\n", nrow(panel_df), ncol(panel_df)))
print(head(panel_df))

Merged panel: 9268 rows, 16 columns
  YEAR CIRI COW DISAP KILL POLPRIS TORT Amnesty State hathaway ITT genocide
1 1949   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
2 1950   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
3 1951   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
4 1952   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
5 1953   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
6 1954   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
  rummel massive_repression executions killing
1      1                  0          0      NA
2      1                  0          0      NA
3      1                  0          0      NA
4      1                  0          0      NA
5      1                  0          1      NA
6      1                  0          0      NA


In [23]:
# 6. Fill structural zeros and handle special cases
# NAs within a dataset's coverage window are true absences, not missing data.
# Fill with 0 only for years the dataset actually covers.

panel_df$genocide[is.na(panel_df$genocide) & panel_df$YEAR >= 1956] = 0
panel_df$rummel[is.na(panel_df$rummel) & panel_df$YEAR <= 1987] = 0
panel_df$massive_repression[is.na(panel_df$massive_repression) & panel_df$YEAR <= 1988] = 0
panel_df$killing[is.na(panel_df$killing) & panel_df$YEAR >= 1989] = 0

# NOTE: 666.001/.002/.003 are Israel sub-codes from COW's treatment of the occupied
# territories — not real country-units, so event indicators are set to NA
for (code in c(666.001, 666.002, 666.003)) {
  panel_df$genocide[panel_df$COW == code] = NA
  panel_df$rummel[panel_df$COW == code] = NA
  panel_df$killing[panel_df$COW == code] = NA
  panel_df$massive_repression[panel_df$COW == code] = NA
}

# Drop rows where every indicator is NA
panel_df = subset(panel_df,
  !is.na(DISAP) | !is.na(KILL) | !is.na(POLPRIS) | !is.na(TORT) |
  !is.na(Amnesty) | !is.na(State) | !is.na(hathaway) | !is.na(ITT) |
  !is.na(genocide) | !is.na(rummel) | !is.na(massive_repression) |
  !is.na(killing) | !is.na(executions)
)

cat(sprintf("Panel after filtering: %d rows, %d unique country-years\n",
            nrow(panel_df), length(unique(paste(panel_df$COW, panel_df$YEAR)))))

print(head(panel_df))

Panel after filtering: 9268 rows, 9268 unique country-years
  YEAR CIRI COW DISAP KILL POLPRIS TORT Amnesty State hathaway ITT genocide
1 1949   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
2 1950   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
3 1951   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
4 1952   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
5 1953   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
6 1954   NA   2    NA   NA      NA   NA      NA    NA       NA  NA       NA
  rummel massive_repression executions killing
1      1                  0          0      NA
2      1                  0          0      NA
3      1                  0          0      NA
4      1                  0          0      NA
5      1                  0          1      NA
6      1                  0          0      NA


In [24]:
# 7. Build panel index vectors for JAGS
# country_idx: which country (1 ... n.country)
# year_idx: position within that country's run (1 ... T_i)
# panel_lens: length of each country's time series

n = nrow(panel_df)

country_idx = integer(n)
year_idx = integer(n)
panel_lens = integer(0)

country_idx[1] = 1
year_idx[1] = 1
panel_count = 1
j = 1

for (i in 2:n) {
  if (panel_df$COW[i] != panel_df$COW[i - 1]) {
    panel_lens[j] = panel_count
    panel_count = 0
    j = j + 1
  }
  panel_count = panel_count + 1
  country_idx[i] = j
  year_idx[i] = panel_count
}
panel_lens[j] = panel_count

# time maps each row to years elapsed since STARTYEAR (t=1 in 1949)
time_idx = panel_df$YEAR - STARTYEAR + 1

cat(sprintf("Countries: %d | Max panel length: %d | Time range: %d to %d\n",
            length(panel_lens), max(panel_lens), min(time_idx), max(time_idx)))

Countries: 204 | Max panel length: 62 | Time range: 1 to 62


In [ ]:
# 8. Build the y matrix (13 indicators fed into JAGS)
# CIRI's four ordinal items (cols 1-4) are shifted +1 so the minimum category = 1,
# required by JAGS's dcat distribution. Remaining items passed through unchanged.

# NOTE: panel_df cols are COW, YEAR, then 13 indicators (cols 3:15) — CIRI was dropped
y_raw = as.matrix(panel_df[, 3:15])
y_dt = matrix(NA, nrow = nrow(y_raw), ncol = ncol(y_raw))

for (i in 1:nrow(y_raw)) {
  for (j in 1:4)  y_dt[i, j] = as.numeric(y_raw[i, j]) + 1
  for (j in 5:13) y_dt[i, j] = as.numeric(y_raw[i, j])
}

colnames(y_dt) = c("DISAP", "KILL", "POLPRIS", "TORT",
                   "Amnesty", "State", "hathaway", "ITT",
                   "genocide", "rummel", "massive_repression",
                   "executions", "killing")

cat(sprintf("y matrix: %d rows x %d columns\n", nrow(y_dt), ncol(y_dt)))
cat(sprintf("  CIRI DISAP range after shift: %d to %d\n",
            min(y_dt[, 1], na.rm = TRUE), max(y_dt[, 1], na.rm = TRUE)))

In [26]:
# 9. Save to data/clean/
dir.create(PATH_DATA_CLEAN, showWarnings = FALSE)

write.csv(panel_df, paste0(PATH_DATA_CLEAN, "merged_panel.csv"), row.names = FALSE)
write.csv(as.data.frame(y_dt), paste0(PATH_DATA_CLEAN, "y_matrix.csv"), row.names = FALSE)

indices_df = data.frame(country = country_idx, year = year_idx, time = time_idx)
write.csv(indices_df, paste0(PATH_DATA_CLEAN, "panel_indices.csv"), row.names = FALSE)
write.csv(data.frame(panel_length = panel_lens), paste0(PATH_DATA_CLEAN, "panel_lengths.csv"), row.names = FALSE)

cat("Saved to data/clean/:\n")
cat("  merged_panel.csv\n  y_matrix.csv\n  panel_indices.csv\n  panel_lengths.csv\n")

Saved to data/clean/:
  merged_panel.csv
  y_matrix.csv
  panel_indices.csv
  panel_lengths.csv
